# RelCheck — LLaVA Correction Test (relcheck_v2)

**Goal:** Test RelCheck on real LLaVA-1.5 captions (no synthetic injection).  
Measures correction quality in isolation and with addendum.

## Methods compared

| ID | Description |
|----|-------------|
| `llava_orig` | LLaVA-1.5-7B original captions (B1 baseline) |
| `llava_corrected` | RelCheck correction only — no fact appending |
| `llava_full` | RelCheck correction + KB fact addendum (full system) |

## Pipeline
1. **Load models** — GroundingDINO (GPU), LLaVA-1.5-7B (4-bit, GPU)
2. **Caption** — LLaVA generates detailed captions (~90 words each)
3. **Build KB** — GroundingDINO detections + Qwen VLM description
4. **Correct** — `relcheck_v2.correction.correct_long_caption` (Session 11 fixes)
5. **Evaluate** — R-POPE LLM-judge on all 3 variants, plus summary table

## Key parameter
Set `N_IMAGES = 50` for a fast validation run (~45 min, ~$0.40 Together.ai).  
Scale to `N_IMAGES = 200` for publication-quality results.


In [ ]:
# ============================================================
# Cell 1 — Install + Setup
# ============================================================
!pip install -q together gdown transformers pillow torch torchvision accelerate bitsandbytes>=0.46.1
!pip install -q spacy open-clip-torch json-repair pysbd rapidfuzz nltk
!python -m spacy download en_core_web_sm -q
!pip install -q nltk
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import os, sys, json, time, random, gc
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import base64

from google.colab import drive
drive.mount("/content/drive")

# ── Clone / pull relcheck_v2 ──────────────────────────────
REPO_DIR = "/content/RelCheck"
if not os.path.exists(f"{REPO_DIR}/.git"):
    import shutil
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.system(f"git clone https://github.com/siddhipatil503/RelCheck {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Settings ─────────────────────────────────────────────
TOGETHER_API_KEY = ""          # <-- paste your Together.ai key
N_IMAGES         = 50          # 50 = fast validation; 200 = publication quality
RANDOM_SEED      = 42

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH      = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"
SAVE_DIR_600     = "/content/drive/MyDrive/RelCheck_Data/run_600"  # reuse KB from here
SAVE_DIR         = "/content/drive/MyDrive/RelCheck_Data/llava_test"

VLM_MODEL    = "Qwen/Qwen3-VL-8B-Instruct"
LLM_MODEL    = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
GDINO_ID     = "IDEA-Research/grounding-dino-tiny"
VITPOSE_ID   = "usyd-community/vitpose-base-simple"
DETECTION_THRESHOLD = 0.25

os.makedirs(SAVE_DIR, exist_ok=True)
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
random.seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Init relcheck_v2 API client ──────────────────────────
import relcheck_v2.api as _api
_api.init_client(TOGETHER_API_KEY)

def save_checkpoint(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def load_checkpoint(path):
    if os.path.exists(path): 
        with open(path) as f: return json.load(f)
    return None

print(f"Device : {DEVICE}")
print(f"Save to: {SAVE_DIR}")
print(f"Images : {N_IMAGES}")


In [ ]:
# ============================================================
# Cell 2 — Load Models (GroundingDINO + ViTPose)
# ============================================================
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    VitPoseForPoseEstimation,
)

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO ready.")

print("Loading ViTPose...")
vitpose_processor = AutoProcessor.from_pretrained(VITPOSE_ID)
vitpose_model = VitPoseForPoseEstimation.from_pretrained(VITPOSE_ID).to(DEVICE)
print("  ViTPose ready.")

# Inject into relcheck_v2.models singletons so get_gdino() reuses them
import relcheck_v2.models as _models
_models._gdino_model     = gdino_model
_models._gdino_processor = gdino_processor
_models.DEVICE           = DEVICE

# Also set detection threshold in config
import relcheck_v2.config as _cfg
_cfg.DETECTION_THRESHOLD = DETECTION_THRESHOLD

print(f"\nAll models ready on {DEVICE}.")


In [ ]:
# ============================================================
# Cell 3 — Load Images + R-Bench Data
# ============================================================
import pathlib, zipfile

# ── Download R-Bench if needed ──
if not os.path.exists(RBENCH_PATH):
    print("Downloading R-Bench...")
    RBENCH_DIR = "/content/R-Bench"
    if not os.path.exists(RBENCH_DIR):
        os.system(f"git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}")
    ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
    if not os.path.exists(f"{RBENCH_DIR}/data"):
        os.system(f"gdown --id 1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH -O {ANNOTATIONS_ZIP} -q")
        with zipfile.ZipFile(ANNOTATIONS_ZIP) as z:
            z.extractall(RBENCH_DIR)
    for f in sorted(pathlib.Path(RBENCH_DIR).rglob("*.json")):
        try:
            data = json.load(open(f))
            if isinstance(data, list) and len(data) > 50:
                normalized = []
                for entry in data:
                    img_file = entry.get("image") or entry.get("img","")
                    img_id   = os.path.splitext(os.path.basename(img_file))[0]
                    q = entry.get("text") or entry.get("question","")
                    a = (entry.get("label") or entry.get("answer","")).lower().strip()
                    if img_id and q and a in ("yes","no"):
                        normalized.append({"img_id": img_id, "question": q, "answer": a})
                if len(normalized) > 100:
                    with open(RBENCH_PATH, "w") as fout:
                        json.dump(normalized, fout)
                    print(f"  Saved {len(normalized)} Q&A pairs.")
                    break
        except: pass

rbench = json.load(open(RBENCH_PATH))

def _get_field(entry, *keys, default=None):
    for k in keys:
        if k in entry: return entry[k]
    return default

img_to_questions = defaultdict(list)
for entry in rbench:
    # Handle both normalized (img_id) and raw (image/img) formats
    img_id = _get_field(entry, "img_id", "img", "image")
    if img_id:
        # Strip path + extension if it's a filename
        img_id = os.path.splitext(os.path.basename(img_id))[0]
    q  = _get_field(entry, "question", "text", "q")
    gt = (_get_field(entry, "answer", "label", "gt") or "").lower().strip()
    if img_id and q and gt in ("yes", "no"):
        img_to_questions[img_id].append({"question": q, "answer": gt})

# ── Load images — only from IDs that have cached LLaVA captions + KB ──
LLAVA_CAPS_PATH = f"{SAVE_DIR}/llava_captions.json"
KB_PATH = f"{SAVE_DIR}/knowledge_bases.json"
_cached_llava = json.load(open(LLAVA_CAPS_PATH)) if os.path.exists(LLAVA_CAPS_PATH) else {}
_cached_kb    = json.load(open(KB_PATH)) if os.path.exists(KB_PATH) else {}

all_rbench_ids = sorted({os.path.splitext(os.path.basename(_get_field(e, "img_id", "img", "image", default="")))[0] for e in rbench} - {""})

# Filter to IDs that have cached LLaVA captions AND KB (skip re-computing)
if _cached_llava:
    available_ids = [i for i in all_rbench_ids if i in _cached_llava and i in _cached_kb]
    print(f"R-Bench IDs: {len(all_rbench_ids)}, with cached LLaVA+KB: {len(available_ids)}")
else:
    available_ids = all_rbench_ids
    print(f"No cached LLaVA captions found — will generate from scratch for {len(available_ids)} IDs")

random.shuffle(available_ids)
selected_ids = available_ids[:N_IMAGES]

images = {}
for img_id in selected_ids:
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(DRIVE_IMAGES_DIR, img_id + ext)
        if os.path.exists(p):
            images[img_id] = Image.open(p).convert("RGB")
            break

print(f"Loaded {len(images)} images, {sum(len(q) for q in img_to_questions.values())} total Q&A pairs")
print(f"Avg questions per image: {np.mean([len(img_to_questions[i]) for i in images]):.1f}")


In [ ]:
# ============================================================
# Cell 4 — LLaVA-1.5-7B Captioning
# ============================================================
from transformers import LlavaForConditionalGeneration, BitsAndBytesConfig

LLAVA_CAPS_PATH = f"{SAVE_DIR}/llava_captions.json"
DESCRIBE_PROMPT = ("Describe this image in detail. Include all objects, their spatial "
                   "positions relative to each other, any actions or interactions taking "
                   "place, and notable attributes like colors and sizes.")

llava_captions = load_checkpoint(LLAVA_CAPS_PATH) or {}

if len(llava_captions) >= len(images):
    print(f"LLaVA captions loaded from cache ({len(llava_captions)}).")
else:
    print("Loading LLaVA-1.5-7B (4-bit)...")
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16)
    llava_model = LlavaForConditionalGeneration.from_pretrained(
        "llava-hf/llava-1.5-7b-hf", quantization_config=bnb,
        device_map="auto", torch_dtype=torch.float16)
    llava_proc = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
    print("LLaVA loaded.")

    todo = [i for i in images if i not in llava_captions]
    print(f"Captioning {len(todo)} images...")

    for idx, img_id in enumerate(todo):
        conversation = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": DESCRIBE_PROMPT},
        ]}]
        prompt_text = llava_proc.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = llava_proc(images=images[img_id], text=prompt_text,
                            return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = llava_model.generate(**inputs, max_new_tokens=256, do_sample=False)
        decoded = llava_proc.decode(out[0], skip_special_tokens=True)
        caption = decoded.split("ASSISTANT:")[-1].strip() if "ASSISTANT:" in decoded else decoded.strip()
        llava_captions[img_id] = caption

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(todo)}] {img_id}: {caption[:80]}...")
            save_checkpoint(llava_captions, LLAVA_CAPS_PATH)

    save_checkpoint(llava_captions, LLAVA_CAPS_PATH)
    del llava_model, llava_proc
    gc.collect(); torch.cuda.empty_cache()
    print("LLaVA unloaded.")

words = [len(c.split()) for c in llava_captions.values() if c]
print(f"Avg caption length: {np.mean(words):.0f} words (min={min(words)}, max={max(words)})")
print(f"All captions >= 30 words: {sum(1 for w in words if w >= 30)}/{len(words)} → correction mode")


In [ ]:
# ============================================================
# Cell 5 — Knowledge Base: Load from 600-run + Gap-Fill Missing
# ============================================================
# 1. Load cached KB from 600-image run (bulk of images already done)
# 2. Build KB for any missing images using GroundingDINO + Qwen VLM
# 3. Save combined KB back to SAVE_DIR

KB_600_PATH = f"{SAVE_DIR_600}/knowledge_bases.json"
KB_PATH = f"{SAVE_DIR}/knowledge_bases.json"

BROAD_CATEGORIES = [
    "person","man","woman","child","boy","girl","dog","cat","bird","horse","animal",
    "car","bicycle","motorcycle","bus","truck","chair","table","bench","couch","bed",
    "food","plate","bowl","cup","bottle","glass","bag","umbrella","phone","book","sign",
    "hat","jacket","vest","helmet","glasses","tree","flower","grass","water",
]

VLM_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe:
- SPATIAL: Where are they relative to each other?
- ACTIONS: What is each person/animal doing?
- ATTRIBUTES: Relevant attributes tied to relationships

Rules: only describe what you can clearly see. Be brief and factual.
Format as a numbered list."""

def _clean_label(label):
    words = label.strip().lower().split()
    while words and words[0] in ('a','an','the'): words = words[1:]
    seen, clean = [], []
    for w in words:
        if w not in seen: seen.append(w); clean.append(w)
    return ' '.join(clean) if clean else label

def detect_objects_kb(image, queries):
    text = ". ".join(queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, threshold=DETECTION_THRESHOLD,
        text_threshold=DETECTION_THRESHOLD, target_sizes=[image.size[::-1]])[0]
    W, H = image.size
    lkey = "text_labels" if "text_labels" in results else "labels"
    raw = [(_clean_label(l), s.item(), [b[0]/W, b[1]/H, b[2]/W, b[3]/H])
           for s, l, b in zip(results["scores"], results[lkey], results["boxes"])]
    # Cap at top-20 most confident to avoid O(n^2) spatial blowup
    raw.sort(key=lambda x: -x[1])
    return raw[:20]

def build_kb_for_image(img_id, image):
    """Build visual KB for a single image using GroundingDINO + Qwen VLM."""
    dets = detect_objects_kb(image, BROAD_CATEGORIES)
    labels = list({d[0] for d in dets})

    # Geometry: pairwise spatial facts from bboxes
    bbox_map = {}
    for label, score, box in sorted(dets, key=lambda x: -x[1]):
        if label not in bbox_map:
            bbox_map[label] = box

    spatial_facts = []
    items = list(bbox_map.items())
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            la, ba = items[i]; lb, bb = items[j]
            cx_a, cy_a = (ba[0]+ba[2])/2, (ba[1]+ba[3])/2
            cx_b, cy_b = (bb[0]+bb[2])/2, (bb[1]+bb[3])/2
            dy = cy_a - cy_b
            dx = cx_a - cx_b
            if abs(dy) > 0.1:
                rel = "below" if dy > 0 else "above"
                spatial_facts.append(f"{la} is {rel} {lb}")
            elif abs(dx) > 0.1:
                rel = "to the right of" if dx > 0 else "to the left of"
                spatial_facts.append(f"{la} is {rel} {lb}")

    # VLM description via Qwen (not Maverick)
    visual_description = ""
    if labels:
        from relcheck_v2.api import vlm_call
        buf = BytesIO()
        image.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode()
        det_list = "\n".join(f"- {l}" for l in labels[:15])
        raw = vlm_call([{"role": "user", "content": [
            {"type": "text", "text": VLM_KB_PROMPT.format(detection_list=det_list)},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]}], max_tokens=300)
        visual_description = raw or ""

    return {
        "detections": [{"label": l, "score": s, "bbox": b} for l, s, b in dets],
        "spatial_facts": spatial_facts,
        "hard_facts": [f"{l} is present" for l in labels],
        "visual_description": visual_description,
    }

# ── Step 1: Load from 600-image run ──
knowledge_bases = load_checkpoint(KB_600_PATH) or {}
n_from_600 = sum(1 for i in images if i in knowledge_bases)
print(f"Loaded KB from 600-image run: {n_from_600}/{len(images)} images covered.")

# ── Step 2: Also check local save dir ──
local_kb = load_checkpoint(KB_PATH) or {}
for img_id in images:
    if img_id not in knowledge_bases and img_id in local_kb:
        knowledge_bases[img_id] = local_kb[img_id]

n_cached = sum(1 for i in images if i in knowledge_bases)
print(f"After local cache: {n_cached}/{len(images)} images covered.")

# ── Step 3: Gap-fill missing images ──
missing = [i for i in images if i not in knowledge_bases]
if missing:
    print(f"\nBuilding KB for {len(missing)} missing images (GroundingDINO + Qwen)...")
    for idx, img_id in enumerate(missing):
        t0 = time.time()
        knowledge_bases[img_id] = build_kb_for_image(img_id, images[img_id])
        elapsed = time.time() - t0
        kb = knowledge_bases[img_id]
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f"  [{idx+1}/{len(missing)}] {img_id}: "
                  f"{len(kb['detections'])} dets, {len(kb['spatial_facts'])} spatial ({elapsed:.1f}s)")
            save_checkpoint(knowledge_bases, KB_PATH)
        time.sleep(0.3)
    save_checkpoint(knowledge_bases, KB_PATH)
    print(f"Gap-fill complete. Saved to {KB_PATH}")
else:
    print("All images have KB — no gap-fill needed.")

print(f"\nKB ready: {sum(1 for i in images if i in knowledge_bases)}/{len(images)} images.")


In [ ]:
# ============================================================
# Cell 6 — RelCheck Correction (3 variants via relcheck_v2)
# ============================================================
# Uses relcheck_v2.correction.correct_long_caption with Session 11 fixes:
#   - MEDIUM confidence gate (lines 180, 265 of _verify.py)
#   - KB-first relation selection + constrained VLM choice
#   - Surgical span editing (never shortens caption)
#
# variant_a: correction only   (include_addendum=False)  ← isolates correction
# variant_b: correction + KB facts (include_addendum=True) ← full system

from relcheck_v2.correction import correct_long_caption

CORR_ONLY_PATH = f"{SAVE_DIR}/llava_corrected.json"

def run_relcheck(caps, save_path, include_addendum, label):
    results = load_checkpoint(save_path) or {}
    todo = [i for i in images if i not in results and caps.get(i)]
    if not todo:
        print(f"{label}: loaded {len(results)} from cache.")
        return results

    print(f"{label}: correcting {len(todo)} captions (addendum={include_addendum})...")
    for idx, img_id in enumerate(todo):
        cap = caps[img_id]
        kb  = knowledge_bases.get(img_id, {})
        t0  = time.time()

        result = correct_long_caption(
            img_id           = img_id,
            caption          = cap,
            kb               = kb,
            pil_image        = images.get(img_id),
            cross_captions   = None,
            include_addendum = include_addendum,
        )

        results[img_id] = {
            "original"  : result.original,
            "corrected" : result.corrected,
            "status"    : result.status,
            "edit_rate" : result.edit_rate,
            "n_triples" : result.n_triples,
            "n_addendum": result.n_addendum,
            "errors"    : [{"claim": e.triple.claim, "confidence": e.confidence.value,
                            "reason": e.reason} for e in (result.errors or [])],
            "time_s"    : round(time.time() - t0, 1),
        }

        if (idx + 1) % 5 == 0 or result.status == "modified":
            r = results[img_id]
            print(f"  [{idx+1}/{len(todo)}] {img_id} → {r['status']} "
                  f"edit={r['edit_rate']:.3f} errs={len(r['errors'])} "
                  f"addendum={r['n_addendum']} ({r['time_s']}s)")
            save_checkpoint(results, save_path)

    save_checkpoint(results, save_path)
    return results

corrected_only = run_relcheck(llava_captions, CORR_ONLY_PATH,
                               include_addendum=False, label="correction-only")
# corrected_full removed — user only needs correction-only variant

# ── Diagnostics: why were captions changed or not? ──
print(f"\n{'='*65}")
print("PIPELINE DIAGNOSTICS (correction-only variant)")
print(f"{'='*65}")
total_triples = sum(v.get("n_triples",0) for v in corrected_only.values())
total_errors  = sum(len(v.get("errors",[])) for v in corrected_only.values())
total_modified = sum(1 for v in corrected_only.values() if v["status"]=="modified")
total_unchanged = sum(1 for v in corrected_only.values() if v["status"]=="unchanged")
print(f"  Total images     : {len(corrected_only)}")
print(f"  Total triples    : {total_triples} ({total_triples/max(len(corrected_only),1):.1f} per image)")
print(f"  Triples → errors : {total_errors} ({total_errors/max(total_triples,1):.1%} flagged)")
print(f"  Images modified  : {total_modified}")
print(f"  Images unchanged : {total_unchanged}")
if total_triples == 0:
    print("\n  ⚠ ZERO TRIPLES EXTRACTED — check if Together.ai API key is set!")
elif total_errors == 0:
    print("\n  ⚠ ZERO ERRORS — VQA may be confirming everything. Check if VLM calls are working.")

# ── Summary ──
print(f"\n{'='*65}")
print(f"{'Variant':<25} {'Modified':>10} {'Avg Edit':>10} {'Addendum':>10}")
print("-" * 65)
for label, data in [("correction-only", corrected_only)]:
    n = len(data)
    n_mod   = sum(1 for v in data.values() if v["status"] == "modified")
    avg_e   = np.mean([v["edit_rate"] for v in data.values()])
    n_add   = sum(v.get("n_addendum", 0) for v in data.values())
    orig_w  = np.mean([len(llava_captions[i].split()) for i in data if llava_captions.get(i)])
    corr_w  = np.mean([len(v["corrected"].split()) for v in data.values() if v.get("corrected")])
    print(f"{label:<25} {n_mod:>4}/{n:<4} ({n_mod/n:.0%})  "
          f"{avg_e:>8.3f}  {n_add:>8}  {orig_w:.0f}→{corr_w:.0f}w")
print()

n_errors = sum(len(v["errors"]) for v in corrected_only.values())
conf_dist = Counter(e["confidence"] for v in corrected_only.values()
                    for e in v["errors"])
print(f"Total errors flagged (correction-only): {n_errors}")
print(f"Confidence distribution: {dict(conf_dist)}")


In [ ]:
# ============================================================
# Cell 7 — R-POPE LLM-Judge Evaluation
# ============================================================
# Evaluates all 3 variants (original, correction-only, correction+addendum)
# using Llama-3.3-70B as a text-only judge (no image access).
# This directly measures caption quality — not downstream VQA strength.

RPOPE_PATH = f"{SAVE_DIR}/rpope_llava.json"

RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""

def rpope_judge(caption, question):
    from relcheck_v2.api import llm_call as _llm
    raw = _llm([{"role": "user", "content":
                 RPOPE_PROMPT.format(caption=caption, question=question)}],
               max_tokens=5)
    if raw:
        rl = raw.lower()
        if "yes" in rl and "no" not in rl: return "yes"
        if "no" in rl: return "no"
    return "unknown"

def compute_metrics(pairs):
    tp = sum(1 for p,g in pairs if p=="yes" and g=="yes")
    fp = sum(1 for p,g in pairs if p=="yes" and g=="no")
    fn = sum(1 for p,g in pairs if p=="no"  and g=="yes")
    tn = sum(1 for p,g in pairs if p=="no"  and g=="no")
    n  = len(pairs)
    acc  = (tp+tn)/n if n else 0
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec  = tp/(tp+fn) if (tp+fn) else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    return {"accuracy":acc,"precision":prec,"recall":rec,"f1":f1,"n":n}

method_captions = {
    "llava_orig"       : llava_captions,
    "llava_corrected"  : {i: v["corrected"] for i,v in corrected_only.items()
                          if v.get("corrected")},
}

rpope_raw = load_checkpoint(RPOPE_PATH) or {}
methods_todo = []
for m in method_captions:
    sample_id = next(iter(images), None)
    if sample_id and img_to_questions.get(sample_id):
        key = f"{m}|||{img_to_questions[sample_id][0]['question']}"
        if key not in rpope_raw.get(sample_id, {}):
            methods_todo.append(m)

if not methods_todo:
    print("All R-POPE results loaded from cache.")
else:
    print(f"Running R-POPE for: {methods_todo}")
    for img_idx, img_id in enumerate(images):
        if img_id not in rpope_raw: rpope_raw[img_id] = {}
        for entry in img_to_questions.get(img_id, [])[:10]:
            q, gt = entry["question"], entry["answer"]
            for method in methods_todo:
                key = f"{method}|||{q}"
                if key not in rpope_raw[img_id]:
                    cap = method_captions[method].get(img_id)
                    if cap:
                        rpope_raw[img_id][key] = {
                            "pred": rpope_judge(cap, q), "gt": gt}
        if (img_idx + 1) % 10 == 0:
            print(f"  [{img_idx+1}/{len(images)}] evaluated")
            save_checkpoint(rpope_raw, RPOPE_PATH)

    save_checkpoint(rpope_raw, RPOPE_PATH)

# ── Compute + display results ──
print(f"\n{'='*72}")
print(f"R-POPE LLM-JUDGE RESULTS — {len(images)} images")
print(f"{'='*72}")
print(f"{'Method':<25} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'N':>6}")
print("-" * 72)

base_acc = None
for method in ["llava_orig", "llava_corrected", "llava_full"]:
    pairs = []
    for img_id, qdict in rpope_raw.items():
        for key, val in qdict.items():
            if key.startswith(f"{method}|||"):
                pairs.append((val["pred"], val["gt"]))
    if not pairs: continue
    m = compute_metrics(pairs)
    delta = f"  (+{(m['accuracy']-base_acc)*100:.1f}%)" if base_acc is not None else ""
    print(f"  {method:<23} {m['accuracy']:>6.1%} {m['precision']:>6.1%} "
          f"{m['recall']:>6.1%} {m['f1']:>6.1%} {m['n']:>5}{delta}")
    if base_acc is None: base_acc = m['accuracy']

print()
print("Key: llava_corrected = correction only (no addendum)")
print("     llava_full      = correction + KB fact appending")
print()
print("If llava_corrected > llava_orig:  correction is adding real signal")
print("If llava_full > llava_corrected:  addendum adds signal on top")
print("If llava_full ≈ llava_corrected:  improvement comes from correction alone")


In [ ]:
# ============================================================
# Cell 8 — Error Analysis + Qualitative Examples
# ============================================================
# Shows which images were corrected and what changed.

print(f"{'='*65}")
print("CORRECTION BREAKDOWN")
print(f"{'='*65}")
n_unchanged = sum(1 for v in corrected_only.values() if v["status"]=="unchanged")
n_modified  = sum(1 for v in corrected_only.values() if v["status"]=="modified")
n_skipped   = sum(1 for v in corrected_only.values() if v["status"]=="skipped")
print(f"  unchanged : {n_unchanged}")
print(f"  modified  : {n_modified}")
print(f"  skipped   : {n_skipped}")

conf_dist = Counter(e["confidence"] for v in corrected_only.values()
                    for e in v["errors"])
print(f"\nConfidence distribution of detected errors: {dict(conf_dist)}")
print("(Both HIGH and MEDIUM trigger corrections — Session 11 fix)")

# ── Show 5 corrected examples ──
print(f"\n{'='*65}")
print("CORRECTED EXAMPLES (correction-only, top 5 by edit rate)")
print(f"{'='*65}")
modified = [(i, v) for i, v in corrected_only.items() if v["status"] == "modified"]
modified.sort(key=lambda x: -x[1]["edit_rate"])

for img_id, v in modified[:5]:
    print(f"\n[{img_id}]  edit_rate={v['edit_rate']:.3f}  errors={len(v['errors'])}")
    print(f"  ORIGINAL : {v['original'][:200]}")
    print(f"  CORRECTED: {v['corrected'][:200]}")
    for e in v["errors"]:
        print(f"  ERROR    : [{e['confidence']}] {e['claim']} — {e['reason'][:80]}")

# ── Addendum contribution ──
if any(v.get("n_addendum",0) > 0 for v in corrected_full.values()):
    print(f"\n{'='*65}")
    print("ADDENDUM CONTRIBUTION (correction+addendum variant)")
    print(f"{'='*65}")
    add_imgs = [(i, v) for i, v in corrected_full.items() if v.get("n_addendum",0) > 0]
    print(f"  Images with facts appended: {len(add_imgs)}/{len(corrected_full)}")
    total_facts = sum(v["n_addendum"] for v in corrected_full.values())
    print(f"  Total facts appended: {total_facts}")
    for img_id, v in add_imgs[:3]:
        orig_len = len(corrected_only[img_id]["corrected"].split())
        full_len = len(v["corrected"].split())
        print(f"  {img_id}: +{v['n_addendum']} facts  ({orig_len}w → {full_len}w)")


In [ ]:
# ============================================================
# Cell 9 — HTML Visualization: Original vs Corrected (10+ images)
# ============================================================
# Generates an interactive HTML report showing:
#   - Image thumbnail
#   - Original vs corrected caption (diff highlighted)
#   - Hallucinated triples + type + confidence + reasoning
#   - KB evidence used for each decision
#   - Per-image R-POPE score change

import difflib
import html as html_mod
from IPython.display import display, HTML as IPyHTML

HTML_PATH = f"{SAVE_DIR}/llava_qualitative.html"

def word_diff_html(original, corrected):
    """Generate highlighted word-level diff."""
    orig_words = original.split()
    corr_words = corrected.split()
    sm = difflib.SequenceMatcher(None, orig_words, corr_words)
    parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == 'equal':
            parts.append(' '.join(orig_words[i1:i2]))
        elif op == 'replace':
            parts.append(f'<span style="background:#ffcccc;text-decoration:line-through;color:#900">{" ".join(orig_words[i1:i2])}</span>')
            parts.append(f'<span style="background:#ccffcc;font-weight:bold;color:#060">{" ".join(corr_words[j1:j2])}</span>')
        elif op == 'delete':
            parts.append(f'<span style="background:#ffcccc;text-decoration:line-through;color:#900">{" ".join(orig_words[i1:i2])}</span>')
        elif op == 'insert':
            parts.append(f'<span style="background:#ccffcc;font-weight:bold;color:#060">{" ".join(corr_words[j1:j2])}</span>')
    return ' '.join(parts)

def img_to_b64_thumb(pil_img, max_size=400):
    img_copy = pil_img.copy()
    img_copy.thumbnail((max_size, max_size))
    buf = BytesIO()
    img_copy.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()

CSS = """
body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
       max-width: 1100px; margin: 0 auto; padding: 20px; background: #f5f5f5; }
h1 { color: #1a1a2e; border-bottom: 3px solid #16213e; padding-bottom: 10px; }
.image-card { background: white; border-radius: 12px; padding: 24px;
              margin-bottom: 30px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }
.image-header { display: flex; gap: 20px; margin-bottom: 16px; }
.thumb { border-radius: 8px; max-width: 350px; height: auto; }
.meta { flex: 1; }
.meta h2 { margin: 0 0 8px 0; color: #16213e; font-size: 18px; }
.status-modified { color: #e94560; font-weight: bold; }
.status-unchanged { color: #0f3460; font-weight: bold; }
.caption-box { background: #fafafa; border-left: 4px solid #ccc;
               padding: 12px 16px; margin: 8px 0; border-radius: 0 8px 8px 0;
               font-size: 14px; line-height: 1.6; }
.caption-box.original { border-left-color: #0f3460; }
.caption-box.corrected { border-left-color: #e94560; }
.caption-box.diff { border-left-color: #f0a500; background: #fffef0; }
.label { font-weight: bold; font-size: 12px; text-transform: uppercase;
         letter-spacing: 1px; margin-bottom: 4px; }
.label.orig { color: #0f3460; }
.label.corr { color: #e94560; }
.label.diff { color: #f0a500; }
.triple-table { width: 100%; border-collapse: collapse; margin: 12px 0; font-size: 13px; }
.triple-table th { background: #16213e; color: white; padding: 8px 12px;
                   text-align: left; font-size: 12px; }
.triple-table td { padding: 8px 12px; border-bottom: 1px solid #eee; }
.triple-table tr:hover { background: #f0f8ff; }
.hallucinated { background: #fff0f0 !important; }
.hallucinated td { color: #900; }
.badge { display: inline-block; padding: 2px 8px; border-radius: 12px;
         font-size: 11px; font-weight: bold; }
.badge-spatial { background: #e8f4fd; color: #0a6abf; }
.badge-action { background: #fef3e2; color: #b45309; }
.badge-attribute { background: #f0e8fd; color: #7c3aed; }
.badge-other { background: #e8e8e8; color: #555; }
.badge-high { background: #fee; color: #c00; }
.badge-medium { background: #fff3cd; color: #856404; }
.badge-low { background: #e8e8e8; color: #555; }
.kb-section { background: #f8f9fa; border-radius: 8px; padding: 12px 16px;
              margin: 12px 0; font-size: 13px; }
.kb-section summary { cursor: pointer; font-weight: bold; color: #16213e; }
.kb-facts { margin: 8px 0; padding-left: 16px; color: #444; }
.rpope-box { display: inline-block; padding: 6px 14px; border-radius: 8px;
             font-size: 13px; margin: 4px; }
.rpope-improved { background: #d4edda; color: #155724; }
.rpope-regressed { background: #f8d7da; color: #721c24; }
.rpope-same { background: #e2e3e5; color: #383d41; }
.summary-bar { display: flex; gap: 20px; margin: 20px 0; flex-wrap: wrap; }
.summary-stat { background: white; border-radius: 8px; padding: 16px 24px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1); text-align: center; }
.summary-stat .num { font-size: 28px; font-weight: bold; color: #16213e; }
.summary-stat .lbl { font-size: 12px; color: #666; text-transform: uppercase; }
"""

# ── Select images: modified first (by edit rate), pad with unchanged ──
modified_imgs = [(i, v) for i, v in corrected_only.items()
                 if v["status"] == "modified" and i in images]
modified_imgs.sort(key=lambda x: -x[1]["edit_rate"])

unchanged_imgs = [(i, v) for i, v in corrected_only.items()
                  if v["status"] == "unchanged" and i in images]

viz_images = modified_imgs[:10]
if len(viz_images) < 10:
    viz_images += unchanged_imgs[:10 - len(viz_images)]

print(f"Generating HTML for {len(viz_images)} images "
      f"({sum(1 for _,v in viz_images if v['status']=='modified')} modified, "
      f"{sum(1 for _,v in viz_images if v['status']=='unchanged')} unchanged)...")

# ── Build per-image R-POPE lookup ──
img_orig_scores = defaultdict(list)
img_corr_scores = defaultdict(list)
for key, entry in rpope_raw.items():
    m = entry.get("method", key.split("|||")[0])
    correct = (entry["pred"] == entry["gt"])
    if m == "llava_orig":
        img_orig_scores[entry["img_id"]].append(correct)
    elif m == "llava_corrected":
        img_corr_scores[entry["img_id"]].append(correct)

# ── Assemble HTML ──
html_parts = []
html_parts.append(f'<!DOCTYPE html>\n<html>\n<head>\n<meta charset="utf-8">\n'
                  f'<title>RelCheck LLaVA-1.5 Qualitative Analysis</title>\n'
                  f'<style>{CSS}</style>\n</head>\n<body>\n'
                  f'<h1>RelCheck &mdash; LLaVA-1.5 Qualitative Analysis</h1>')

# Summary bar
n_total = len(corrected_only)
n_mod = sum(1 for v in corrected_only.values() if v["status"] == "modified")
n_unch = n_total - n_mod

orig_pairs = results_by_method.get("llava_orig", [])
corr_pairs = results_by_method.get("llava_corrected", [])
orig_acc = sum(1 for p, g in orig_pairs if p == g) / max(len(orig_pairs), 1)
corr_acc = sum(1 for p, g in corr_pairs if p == g) / max(len(corr_pairs), 1)
delta = corr_acc - orig_acc

html_parts.append(
    f'<div class="summary-bar">'
    f'<div class="summary-stat"><div class="num">{n_total}</div><div class="lbl">Images Tested</div></div>'
    f'<div class="summary-stat"><div class="num">{n_mod}</div><div class="lbl">Modified</div></div>'
    f'<div class="summary-stat"><div class="num">{n_unch}</div><div class="lbl">Unchanged</div></div>'
    f'<div class="summary-stat"><div class="num">{orig_acc:.1%}</div><div class="lbl">R-POPE Original</div></div>'
    f'<div class="summary-stat"><div class="num">{corr_acc:.1%}</div><div class="lbl">R-POPE Corrected</div></div>'
    f'<div class="summary-stat"><div class="num">{delta:+.1%}</div><div class="lbl">Delta</div></div>'
    f'</div>'
)

SPATIAL_KW = {"on","in","under","above","below","left","right","behind","front",
              "near","next","between","inside","outside","beside","top","bottom"}
ACTION_KW = {"holding","riding","wearing","eating","sitting","standing","walking",
             "running","playing","carrying","reading","looking","using","driving"}

for img_id, v in viz_images:
    original = v.get("original", llava_captions.get(img_id, ""))
    corrected = v.get("corrected", original)
    status = v["status"]
    edit_rate = v.get("edit_rate", 0)
    errors = v.get("errors", [])
    kb = knowledge_bases.get(img_id, {})

    # Image thumbnail
    if img_id in images:
        b64_thumb = img_to_b64_thumb(images[img_id])
        img_tag = f'<img class="thumb" src="data:image/jpeg;base64,{b64_thumb}" alt="{img_id}">'
    else:
        img_tag = '<div style="width:350px;height:250px;background:#ddd;border-radius:8px"></div>'

    status_class = "status-modified" if status == "modified" else "status-unchanged"

    # Per-image R-POPE
    oc = img_orig_scores.get(img_id, [])
    cc = img_corr_scores.get(img_id, [])
    orig_score = sum(oc) / max(len(oc), 1) if oc else None
    corr_score = sum(cc) / max(len(cc), 1) if cc else None

    rpope_html = ""
    if orig_score is not None and corr_score is not None:
        if corr_score > orig_score:
            cls = "rpope-improved"
        elif corr_score < orig_score:
            cls = "rpope-regressed"
        else:
            cls = "rpope-same"
        rpope_html = f'<span class="rpope-box {cls}">R-POPE: {orig_score:.0%} &rarr; {corr_score:.0%}</span>'

    html_parts.append(
        f'<div class="image-card">'
        f'<div class="image-header">{img_tag}'
        f'<div class="meta"><h2>{html_mod.escape(img_id)}</h2>'
        f'<p><span class="{status_class}">{status.upper()}</span>'
        f' &nbsp; edit_rate={edit_rate:.3f}'
        f' &nbsp; errors={len(errors)}'
        f' {rpope_html}</p></div></div>'
        f'<div class="label orig">Original LLaVA Caption</div>'
        f'<div class="caption-box original">{html_mod.escape(original[:500])}</div>'
        f'<div class="label corr">Corrected Caption</div>'
        f'<div class="caption-box corrected">{html_mod.escape(corrected[:500])}</div>'
    )

    if status == "modified":
        diff_html = word_diff_html(original, corrected)
        html_parts.append(
            f'<div class="label diff">Diff (red=removed, green=added)</div>'
            f'<div class="caption-box diff">{diff_html}</div>'
        )

    # Errors table
    if errors:
        rows = '<tr><th>Hallucinated Claim</th><th>Type</th><th>Confidence</th><th>Reasoning</th></tr>'
        for e in errors:
            claim = html_mod.escape(e.get("claim", ""))
            conf = e.get("confidence", "?")
            reason = html_mod.escape(e.get("reason", "")[:250])
            claim_lower = claim.lower()
            if any(kw in claim_lower for kw in SPATIAL_KW):
                type_badge = '<span class="badge badge-spatial">SPATIAL</span>'
            elif any(kw in claim_lower for kw in ACTION_KW):
                type_badge = '<span class="badge badge-action">ACTION</span>'
            else:
                type_badge = '<span class="badge badge-attribute">ATTRIBUTE</span>'
            conf_badge = f'<span class="badge badge-{conf.lower()}">{conf}</span>'
            rows += (f'<tr class="hallucinated"><td>{claim}</td>'
                     f'<td>{type_badge}</td><td>{conf_badge}</td><td>{reason}</td></tr>')
        html_parts.append(f'<table class="triple-table">{rows}</table>')
    else:
        html_parts.append('<p style="color:#666;font-style:italic">No errors detected &mdash; caption passed verification.</p>')

    # KB evidence (collapsible)
    hard = kb.get("hard_facts", [])
    spatial = kb.get("spatial_facts", [])
    visual = kb.get("visual_description", "")
    html_parts.append(
        f'<details class="kb-section">'
        f'<summary>Visual Knowledge Base ({len(hard)} objects, {len(spatial)} spatial facts)</summary>'
        f'<div class="kb-facts">'
        f'<strong>Detected Objects:</strong> {", ".join(str(h) for h in hard[:15]) if hard else "None"}<br>'
        f'<strong>Spatial Facts:</strong> {"; ".join(str(s) for s in spatial[:8]) if spatial else "None"}<br>'
        f'<strong>VLM Description:</strong> {html_mod.escape(str(visual or "None")[:300])}'
        f'</div></details>'
    )

    html_parts.append('</div>')

html_parts.append(
    '<footer style="text-align:center;color:#999;padding:20px;font-size:12px">'
    'Generated by RelCheck &mdash; CS298 Master&#39;s Project &mdash; Siddhi Patil, SJSU 2026'
    '</footer></body></html>'
)

full_html = '\n'.join(html_parts)
with open(HTML_PATH, 'w') as f:
    f.write(full_html)

print(f"Saved qualitative report: {HTML_PATH}")
print(f"   File size: {len(full_html)/1024:.0f} KB")
print(f"   Images shown: {len(viz_images)}")

display(IPyHTML(f'<div style="max-height:800px;overflow-y:auto">{full_html}</div>'))
